# exp_022 — merge pass-4 adapter + push merged model

**Use this when:** GRPO pass 4 training finished on DSMLP but the final merge-and-push step didn't run (here: the inline fp16 merge crashed on the DSMLP 5GB PVC quota — `OSError: [Errno 122] Disk quota exceeded` writing `merged_final/chat_template.jinja`). The trained adapter is safe on HF Hub; this notebook does the merge on Kaggle instead.

**Inputs (HF Hub):**
- Base: `TrevorDuong/qwen3-4b-thinking-grpo-pass3` (already merged, single safetensors)
- Adapter: `TrevorDuong/qwen3-4b-thinking-grpo-pass4-ckpt/checkpoint-LATEST` (final step, full epoch)

**Output (HF Hub):**
- `TrevorDuong/qwen3-4b-thinking-grpo-pass4` — fully merged, vLLM-loadable, drop-in for exp_023 stage-1 inference (or future pass-4 inference). Pushed **public** so the Kaggle inference notebook (no HF auth) can read it without a 401.

**Runtime:** ~10 min on Kaggle T4 x1, internet ON.

**Prereq:** Add a Kaggle Secret named `HF_TOKEN` (write-scope HF token) and attach it to this notebook.

**To re-use for a different checkpoint:** change `SUBFOLDER` below (e.g. `checkpoint-60`).

In [ ]:
# Cell 1 — install deps. ~30s on Kaggle.
!pip install -q "transformers>=4.45,<5" "peft>=0.13" accelerate safetensors

In [ ]:
# Cell 2 — merge and push.
import os, torch
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE      = "TrevorDuong/qwen3-4b-thinking-grpo-pass4"
ADAPTR    = "TrevorDuong/qwen3-4b-thinking-grpo-pass4-ckpt"
TARGET    = "TrevorDuong/qwen3-4b-thinking-grpo-pass4"
SUBFOLDER = "checkpoint-LATEST"

print(f"Loading base: {BASE}")
base = AutoModelForCausalLM.from_pretrained(
    BASE,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"Loading adapter: {ADAPTR}/{SUBFOLDER}")
model = PeftModel.from_pretrained(base, ADAPTR, subfolder=SUBFOLDER)

print("Merging adapter into base weights ...")
model = model.merge_and_unload()

print(f"Pushing merged model to: {TARGET} (public)")
model.push_to_hub(TARGET, safe_serialization=True, private=False)

print("Pushing tokenizer too ...")
tok = AutoTokenizer.from_pretrained(BASE)
tok.push_to_hub(TARGET, private=False)

print("Done!")
print(f"\u2192 https://huggingface.co/{TARGET}")

In [ ]:
# Cell 3 (optional) — verify the push.
import requests
tok = os.environ["HF_TOKEN"]
r = requests.get(
    f"https://huggingface.co/api/models/{TARGET}",
    headers={"Authorization": f"Bearer {tok}"},
    timeout=30,
)
d = r.json()
print("lastModified:", d.get("lastModified"))
print("files:")
for f in d.get("siblings", [])[:20]:
    print(" -", f["rfilename"])